(course-core-08)=

# Module 8: Extracting Molecular Attributes

**Welcome back, Apprentice Master.** In [Module 7: Selection Mechanism](../00_Common_Core/07_Selection_Basics.ipynb), you mastered how to build query expressions to target specific subsets of a molecular system. Now, we couple those selection queries with MolSysMT's primary extraction engine: **`msm.get()`**.

While `msm.info()` is designed for human inspection via formatted tables, `msm.get()` is engineered for programmatic data extraction. It converts internal data structures into clean Python primitives, lists, NumPy arrays, and physical quantities that can be passed directly to data science libraries like `numpy`, `scipy`, `pandas`, or `scikit-learn`.

(course-core-08-learning-outcomes)=
```{admonition} Learning Outcomes
:class: dropdown learning-outcomes

By the end of this module, you will be able to:
- Extract single attributes into direct Python primitives and NumPy arrays.
- Extract multiple attributes simultaneously using structured tuple unpacking.
- Retrieve spatial coordinate tensors with shape `(n_structures, n_atoms, 3)`.
- Extract periodic box vectors, angles, and volumes.
- Combine selection queries (`selection=...`) and element scoping (`element=...`) inside `msm.get()`.
```

### 1. Extracting Single Attributes

Let's begin by importing MolSysMT and NumPy, and loading our T4 Lysozyme demonstration system.

In [1]:
import molsysmt as msm
from molsysmt import systems
import numpy as np

# Load T4 Lysozyme file
lysozyme = systems['T4 lysozyme L99A']['181l.bcif.gz']

When you request a single attribute from `msm.get()`, it returns that property directly without wrapping it in a tuple.

Let's query the names of the first 5 atoms and the total number of atoms in the system:

In [2]:
# Get atom names for the first 5 atoms
names = msm.get(lysozyme, selection=[0, 1, 2, 3, 4], atom_name=True)
print(f"First 5 atom names: {names}")

# Get the total number of atoms
n_atoms = msm.get(lysozyme, element='system', n_atoms=True)
print(f"Total system atoms: {n_atoms}")

First 5 atom names: ['N', 'CA', 'C', 'O', 'CB']
Total system atoms: 1441


:::{hint}
:class: dropdown
**msm.get()**: Form-agnostic attribute extraction engine. Converts internal data into Python lists, NumPy arrays, or scalar primitives. See API doc: {func}`molsysmt.basic.get`.
:::

### 2. Extracting Multiple Attributes

When you request multiple attributes in a single `msm.get()` call, it returns a tuple containing the requested properties in the exact order specified in your function call.

In [3]:
# Extract atom_id, atom_name, and atom_type for specific atom indices
ids, names, types = msm.get(lysozyme, selection=[10, 20, 30], atom_id=True, atom_name=True, atom_type=True)

for atom_id, name, atom_type in zip(ids, names, types):
    print(f"ID: {atom_id:>4} | Name: {name:<4} | Type: {atom_type}")

ID:   11 | Name: C    | Type: C
ID:   21 | Name: CB   | Type: C
ID:   31 | Name: CD1  | Type: C


### 3. Extracting Coordinate Tensors & Spatial Geometry

Coordinates are the most frequently extracted structural attribute. As established in Module 2, coordinates are returned as a **3D NumPy array** with shape `(n_structures, n_atoms, 3)`.

Let's extract the coordinates of all protein atoms and compute their geometric center using `numpy.mean()`:

In [4]:
# Extract coordinates of all protein atoms
coords = msm.get(lysozyme, selection='molecule_type == "protein"', coordinates=True)

print(f"Coordinates array shape: {coords.shape} (n_structures, n_atoms, spatial:x,y,z)")

# Compute geometric center of protein atoms across structure 0
center = np.mean(coords[0], axis=0)
print(f"Geometric center (X, Y, Z) in nanometers: {center}")

Coordinates array shape: (1, 1289, 3) (n_structures, n_atoms, spatial:x,y,z)
Geometric center (X, Y, Z) in nanometers: [3.4858737781225804 1.1384903801396447 0.9543625290923174] nanometer


### 4. Extracting Periodic Box Properties

When a molecular system includes periodic boundary conditions, `msm.get()` can derive its box vectors, lengths, angles, and volume:

In [5]:
# Extract box lengths, angles, and volume
lengths, angles, volume = msm.get(
    lysozyme,
    element='system',
    box_lengths=True,
    box_angles=True,
    box_volume=True
)

print(f"Box lengths: {lengths}")
print(f"Box angles : {angles}")
print(f"Box volume : {volume}")

Box lengths: [[6.09 6.09 9.7]] nanometer
Box angles : [[1.570796 1.570796 2.094395]] radian
Box volume : [311.55659621309997] nanometer ** 3


--- 

### 🏆 Challenge 8: The Data Scientist

1. Load the **T4 Lysozyme** system (`systems['T4 lysozyme L99A']['181l.bcif.gz']`).
2. Extract the coordinates of all **Nitrogen** atoms (`selection='atom_name == "N"'`).
3. Use `numpy.mean()` on the extracted coordinates array to find their center of geometry.
4. Extract the `group_name` and `group_id` for those Nitrogen atoms and print the first 5 entries.

Now that you can extract raw numerical datasets programmatically, you must ensure that physical quantities maintain mathematical safety. In [Module 9: Unit Safety with PyUnitWizard](../00_Common_Core/09_Unit_Safety.ipynb), we will explore physical unit management.

```{key-takeaway}
The `msm.get()` function is the workhorse of programmatic data extraction, converting molecular systems into standard Python lists, NumPy arrays, and physical quantities ready for numerical analysis.
```

(course-core-08-see-also)=
:::{seealso}
:class: dropdown
**API Documentation for Functions in this Module:**
- {func}`molsysmt.basic.get` — Form-agnostic attribute extraction engine.

**Related Course Modules & Guides:**
- Previous Module: [Module 7: Selection Mechanism](../00_Common_Core/07_Selection_Basics.ipynb)
- Next Module: [Module 9: Unit Safety with PyUnitWizard](../00_Common_Core/09_Unit_Safety.ipynb)
- User Guide: {ref}`user-foundations`
:::